导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True


本节重新整理为一个真正可计算的“传播效应三联体”：

- 星际介质中的散射与相干损失；
- 电离层中的色散相位、群时延与法拉第旋转；
- 对流层中的非色散路径涨落与高频相干度下降。

旧版这一章包含不少外部图片和可选依赖；本次改写后，所有示例都由 notebook 自己生成。这样学生看到的不再只是“有很多传播效应”，而是能真正读出数量级、频率依赖和它们对可见度/成像的直接后果。


***


## 7.7 传播效应：信号在到达望远镜之前经历了什么

天空电磁波在进入馈源之前，通常已经穿过了多个传播介质。若把这些效应写进 RIME，它们可以被视作位于“天空”和“天线”之间的外部 Jones 项。虽然不同教材对符号命名并不统一，但物理上大体可分为两类：

- **色散介质**：响应强烈依赖频率，例如电离层等离子体；
- **近似非色散介质**：对相位的影响主要表现为几何路径变化，例如对流层湿延迟（远离强吸收线时）。

另一方面，星际介质的湍流会把原本紧致的源散射成角尺寸更大的表观图像，从而降低分辨率并抹掉长基线的相干性。


In [ ]:
c = 299792458.0


def phase_structure_function(baseline_m, r0_m, beta=5.0 / 3.0):
    return (np.asarray(baseline_m) / r0_m) ** beta


def mutual_coherence_from_structure(Dphi):
    return np.exp(-0.5 * np.asarray(Dphi))


def elliptical_gaussian(nx, ny, sigma_x_pix, sigma_y_pix, angle_deg=0.0, center=None):
    if center is None:
        cx = (nx - 1) / 2.0
        cy = (ny - 1) / 2.0
    else:
        cx, cy = center

    x = np.arange(nx) - cx
    y = np.arange(ny) - cy
    X, Y = np.meshgrid(x, y)
    angle = np.deg2rad(angle_deg)
    X_rot = X * np.cos(angle) + Y * np.sin(angle)
    Y_rot = -X * np.sin(angle) + Y * np.cos(angle)
    return np.exp(-0.5 * ((X_rot / sigma_x_pix) ** 2 + (Y_rot / sigma_y_pix) ** 2))


def fft_convolve2d(image, kernel):
    return np.real(
        np.fft.ifft2(np.fft.fft2(image) * np.fft.fft2(np.fft.ifftshift(kernel)))
    )


def ionospheric_phase_rad(delta_tec_tecu, freq_hz):
    return -8.44797245e9 * np.asarray(delta_tec_tecu) / np.asarray(freq_hz)


def ionospheric_group_delay_s(delta_tec_tecu, freq_hz):
    return 40.3e16 * np.asarray(delta_tec_tecu) / (c * np.asarray(freq_hz) ** 2)


def faraday_rotation_rad(rm_rad_m2, freq_hz):
    return rm_rad_m2 * (c / np.asarray(freq_hz)) ** 2


def synthetic_tec_screen(x_km, y_km):
    X, Y = np.meshgrid(x_km, y_km)
    tec = 0.06 * np.sin(2.0 * np.pi * X / 18.0)
    tec += 0.04 * np.cos(2.0 * np.pi * Y / 11.0)
    tec += 0.03 * np.sin(2.0 * np.pi * (X + 0.7 * Y) / 25.0)
    return tec


def sample_screen_nearest(screen, x_axis, y_axis, points_xy):
    ix = np.clip(np.searchsorted(x_axis, points_xy[:, 0]), 1, len(x_axis) - 1)
    iy = np.clip(np.searchsorted(y_axis, points_xy[:, 1]), 1, len(y_axis) - 1)
    return screen[iy, ix]


def wet_delay_from_pwv_mm(pwv_mm, scale_mm_delay_per_mm_pwv=6.3):
    return scale_mm_delay_per_mm_pwv * np.asarray(pwv_mm) * 1e-3


def phase_from_path_delay_rad(path_delay_m, freq_hz):
    return 2.0 * np.pi * np.asarray(freq_hz) * np.asarray(path_delay_m) / c


### 7.7.1 星际介质：相位结构函数、相干损失与散射展宽

若传播介质可以近似为一张薄相位屏（thin phase screen），则一个非常核心的统计量是相位结构函数：

$$
D_\phi(\rho) = \left\langle \left[\phi(\mathbf{r}+\boldsymbol{\rho}) - \phi(\mathbf{r})\right]^2 \right\rangle.
$$

在 Kolmogorov 湍流近似下，常见的标度关系是 $D_\phi(\rho) \propto \rho^{5/3}$。它直接决定基线相干度：

$$
\gamma(\rho) \approx \exp\left[-\frac{1}{2}D_\phi(\rho)\right].
$$

因此，只要散射屏让大尺度基线上的相位涨落足够大，长基线可见度就会被迅速洗掉。


In [ ]:
baselines_km = np.logspace(-1, 3, 300)
r_diff_1ghz_km = 5.0
r_diff_5ghz_km = r_diff_1ghz_km * 5.0 ** (6.0 / 5.0)

Dphi_1ghz = phase_structure_function(baselines_km * 1e3, r_diff_1ghz_km * 1e3)
Dphi_5ghz = phase_structure_function(baselines_km * 1e3, r_diff_5ghz_km * 1e3)

coh_1ghz = mutual_coherence_from_structure(Dphi_1ghz)
coh_5ghz = mutual_coherence_from_structure(Dphi_5ghz)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].loglog(baselines_km, Dphi_1ghz, label="1 GHz", lw=2.0)
axes[0].loglog(baselines_km, Dphi_5ghz, label="5 GHz", lw=2.0)
axes[0].axhline(1.0, color="black", ls="--", lw=1.0)
axes[0].set_xlabel("baseline [km]")
axes[0].set_ylabel("phase structure function")
axes[0].set_title("Kolmogorov-like phase structure function")
axes[0].legend()

axes[1].semilogx(baselines_km, coh_1ghz, label="1 GHz", lw=2.0)
axes[1].semilogx(baselines_km, coh_5ghz, label="5 GHz", lw=2.0)
axes[1].set_xlabel("baseline [km]")
axes[1].set_ylabel("mutual coherence")
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_title("Long baselines decorrelate first")
axes[1].legend()

fig.tight_layout()
plt.show()
plt.close(fig)

baseline_test_km = 10.0
coh_test_1 = mutual_coherence_from_structure(
    phase_structure_function(baseline_test_km * 1e3, r_diff_1ghz_km * 1e3)
)
coh_test_5 = mutual_coherence_from_structure(
    phase_structure_function(baseline_test_km * 1e3, r_diff_5ghz_km * 1e3)
)

print(f"在 {baseline_test_km:.0f} km 基线上：")
print(f"  1 GHz coherence ≈ {coh_test_1:.3f}")
print(f"  5 GHz coherence ≈ {coh_test_5:.3f}")
print(f"  1 GHz 的相位相干尺度 r_diff ≈ {r_diff_1ghz_km:.1f} km")
print(f"  5 GHz 的相位相干尺度 r_diff ≈ {r_diff_5ghz_km:.1f} km")


这一步告诉我们：散射不是“让图像稍微模糊一点”这么简单。它首先会在可见度域里把长基线相干性打碎，然后才体现为像域中的展宽与模糊。

在强散射近似下，紧致源的表观角尺寸通常会随波长迅速增大，常见经验关系近似为 $\theta_{\rm scatt} \propto \lambda^2$。下面用一个自造的紧致源图像，演示这种随频率变化的展宽后果。


In [ ]:
nx = ny = 201
source = elliptical_gaussian(nx, ny, sigma_x_pix=2.2, sigma_y_pix=1.4, angle_deg=15.0)
source /= source.max()

scatter_freqs_ghz = [1.0, 2.0, 5.0]
scattered_images = []
scatter_summary = []

for nu_ghz in scatter_freqs_ghz:
    scale = (1.0 / nu_ghz) ** 2
    sigma_major = 7.0 * scale
    sigma_minor = 3.0 * scale
    kernel = elliptical_gaussian(nx, ny, sigma_major, sigma_minor, angle_deg=30.0)
    kernel /= kernel.sum()
    scattered = fft_convolve2d(source, kernel)
    scattered /= scattered.max()
    scattered_images.append(scattered)
    scatter_summary.append((nu_ghz, 2.355 * sigma_major, 2.355 * sigma_minor))

fig, axes = plt.subplots(2, 2, figsize=(10, 9))

axes[0, 0].imshow(source, origin="lower", cmap="magma")
axes[0, 0].set_title("Intrinsic compact source")
axes[0, 0].set_xlabel("x [pix]")
axes[0, 0].set_ylabel("y [pix]")

for ax, nu_ghz, image in zip(axes.flat[1:], scatter_freqs_ghz, scattered_images):
    ax.imshow(image, origin="lower", cmap="magma")
    ax.set_title(f"Scattered image at {nu_ghz:.1f} GHz")
    ax.set_xlabel("x [pix]")
    ax.set_ylabel("y [pix]")

fig.tight_layout()
plt.show()
plt.close(fig)

print("示例中采用的散射核 FWHM（像素单位）：")
for nu_ghz, fwhm_major, fwhm_minor in scatter_summary:
    print(
        f"  {nu_ghz:.1f} GHz -> major ≈ {fwhm_major:.2f} pix, minor ≈ {fwhm_minor:.2f} pix"
    )


这里的卷积核是“集合平均散射”的近似模型，而不是完整的波动传播解。但它已经足以抓住两条关键经验：

- 低频时表观源尺寸显著变大；
- 散射往往是各向异性的，因此图像不仅变宽，还可能沿某个方向被拉长。

对于要依靠超长基线解析极紧致结构的观测，例如黑洞阴影或脉冲星散射测量，这种展宽绝不只是一个次要修正项。


### 7.7.2 电离层：色散相位、群时延与法拉第旋转

电离层是一层稀薄等离子体，因此它的核心特征是**色散**。对射电干涉最重要的三个量分别是：

1. 差分 TEC 带来的相位项，近似满足
   $$
   \phi_{\rm ion} \approx -8.45 \frac{\Delta\mathrm{TEC}\,[\mathrm{TECU}]}{\nu_{\mathrm{GHz}}}\;\mathrm{rad};
   $$
2. 与之对应的群时延，近似满足
   $$
   \tau_{\rm ion} \approx 1.34 \frac{\Delta\mathrm{TEC}\,[\mathrm{TECU}]}{\nu_{\mathrm{GHz}}^2}\;\mathrm{ns};
   $$
3. 对线偏振面的法拉第旋转
   $$
   \chi_F = \mathrm{RM}\,\lambda^2.
   $$

这三项里，相位与时延主要破坏可见度相干性，而法拉第旋转会直接污染极化分析。


In [ ]:
iono_freqs_hz = np.logspace(np.log10(50e6), np.log10(5e9), 400)
delta_tec_tecu = 0.1
rm_rad_m2 = 1.0

iono_phase_abs = np.abs(ionospheric_phase_rad(delta_tec_tecu, iono_freqs_hz))
iono_delay_ns = ionospheric_group_delay_s(delta_tec_tecu, iono_freqs_hz) * 1e9
faraday_deg = np.abs(np.rad2deg(faraday_rotation_rad(rm_rad_m2, iono_freqs_hz)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].loglog(iono_freqs_hz / 1e6, iono_phase_abs, label="|phase| for 0.1 TECU", lw=2.0)
axes[0].loglog(iono_freqs_hz / 1e6, iono_delay_ns, label="group delay [ns]", lw=2.0)
axes[0].set_xlabel("frequency [MHz]")
axes[0].set_ylabel("magnitude")
axes[0].set_title("Dispersive ionospheric phase and delay")
axes[0].legend()

axes[1].loglog(iono_freqs_hz / 1e6, faraday_deg, color="tab:green", lw=2.0)
axes[1].set_xlabel("frequency [MHz]")
axes[1].set_ylabel("|Faraday rotation| [deg]")
axes[1].set_title("Faraday rotation grows rapidly toward low frequency")

fig.tight_layout()
plt.show()
plt.close(fig)

for nu_hz in [150e6, 1.4e9]:
    phase = ionospheric_phase_rad(delta_tec_tecu, nu_hz)
    delay_ns = ionospheric_group_delay_s(delta_tec_tecu, nu_hz) * 1e9
    chi_deg = np.rad2deg(faraday_rotation_rad(rm_rad_m2, nu_hz))
    print(
        f"nu = {nu_hz / 1e6:7.1f} MHz -> phase = {phase:7.3f} rad, "
        f"delay = {delay_ns:6.3f} ns, Faraday = {chi_deg:7.2f} deg"
    )


从这些量级可以看出一个非常清楚的结论：到了低频，电离层绝不是“稍微有点影响”。哪怕只有 `0.1 TECU` 的差分电子含量，也足以在百兆赫兹量级产生数弧度相位误差，并把线偏振面旋转上百角度。

更麻烦的是，电离层并不是一层静止、平滑、均匀的介质，而更像一张随时间漂移的二维屏幕。下面用一张合成 TEC 屏幕，看看不同频率下阵列各天线会看到怎样的相位图样。


In [ ]:
x_km = np.linspace(-10.0, 10.0, 181)
y_km = np.linspace(-10.0, 10.0, 181)
tec_screen = synthetic_tec_screen(x_km, y_km)

rng = np.random.default_rng(42)
antenna_xy_km = rng.uniform(-8.0, 8.0, size=(14, 2))
antenna_tec = sample_screen_nearest(tec_screen, x_km, y_km, antenna_xy_km)

phase_150 = ionospheric_phase_rad(antenna_tec - antenna_tec[0], 150e6)
phase_1400 = ionospheric_phase_rad(antenna_tec - antenna_tec[0], 1.4e9)

wrapped_150 = np.angle(np.exp(1.0j * phase_150))
wrapped_1400 = np.angle(np.exp(1.0j * phase_1400))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

im0 = axes[0].imshow(
    tec_screen,
    origin="lower",
    extent=[x_km[0], x_km[-1], y_km[0], y_km[-1]],
    cmap="coolwarm",
)
axes[0].scatter(antenna_xy_km[:, 0], antenna_xy_km[:, 1], s=30, color="black")
axes[0].set_title("Synthetic differential TEC screen [TECU]")
axes[0].set_xlabel("x [km]")
axes[0].set_ylabel("y [km]")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].scatter(
    antenna_xy_km[:, 0], antenna_xy_km[:, 1], c=wrapped_150, s=120, cmap="twilight", vmin=-np.pi, vmax=np.pi
)
axes[1].set_title("Wrapped antenna phase at 150 MHz")
axes[1].set_xlabel("x [km]")
axes[1].set_ylabel("y [km]")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].scatter(
    antenna_xy_km[:, 0], antenna_xy_km[:, 1], c=wrapped_1400, s=120, cmap="twilight", vmin=-np.pi, vmax=np.pi
)
axes[2].set_title("Wrapped antenna phase at 1.4 GHz")
axes[2].set_xlabel("x [km]")
axes[2].set_ylabel("y [km]")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"TEC 屏幕标准差 ≈ {tec_screen.std():.3f} TECU")
print(f"150 MHz 时阵列相位 RMS ≈ {np.std(phase_150):.3f} rad")
print(f"1.4 GHz 时阵列相位 RMS ≈ {np.std(phase_1400):.3f} rad")


这里可以直接读出一个实践经验：同一张 TEC 屏幕，在低频上可能已经把阵列推到“满屏相位包裹”的状态，而在 GHz 量级则只表现为缓慢的相位坡度。这也是低频阵列为什么往往更依赖方向依赖校准与电离层屏建模。


### 7.7.3 对流层：湿延迟近似非色散，但高频相位最怕它

对流层中的主要麻烦来自水汽引起的额外路径长度。远离强吸收线时，这个路径误差近似是非色散的，因此它不像电离层那样出现 $\nu^{-1}$ 或 $\nu^{-2}$ 的色散项；但相位本身满足

$$
\phi_{\rm trop} = \frac{2\pi \nu}{c} \Delta L,
$$

因而**频率越高，相同的路径误差会变成越大的相位误差**。这就是毫米波和亚毫米波观测为什么极度依赖水汽辐射计、快切换和良好天气。

一个常用的工程近似是：`1 mm PWV` 对应约 `6.3 mm` 的湿延迟路径。下面先看固定水汽涨落下的频率响应。


In [ ]:
tropo_freqs_hz = np.linspace(10e9, 350e9, 400)
pwv_rms_mm = 0.05
wet_delay_rms_m = wet_delay_from_pwv_mm(pwv_rms_mm)
tropo_phase_rms = phase_from_path_delay_rad(wet_delay_rms_m, tropo_freqs_hz)
tropo_coherence = np.exp(-0.5 * tropo_phase_rms**2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(tropo_freqs_hz / 1e9, tropo_phase_rms, lw=2.0)
axes[0].set_xlabel("frequency [GHz]")
axes[0].set_ylabel("phase RMS [rad]")
axes[0].set_title("Same wet delay causes larger phase at higher frequency")

axes[1].plot(tropo_freqs_hz / 1e9, tropo_coherence, lw=2.0, color="tab:orange")
axes[1].set_xlabel("frequency [GHz]")
axes[1].set_ylabel("coherence loss factor")
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_title("Coherence drops rapidly toward mm/sub-mm")

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"PWV RMS = {pwv_rms_mm:.3f} mm -> wet delay RMS ≈ {wet_delay_rms_m * 1e3:.3f} mm")
for nu_hz in [22e9, 100e9, 230e9]:
    phase = phase_from_path_delay_rad(wet_delay_rms_m, nu_hz)
    coherence = np.exp(-0.5 * phase**2)
    print(f"  {nu_hz / 1e9:6.1f} GHz -> phase RMS = {phase:6.3f} rad, coherence ≈ {coherence:.3f}")


这一步强调的是“非色散不等于不严重”。对流层的难点并不是复杂色散，而是它在高频上把极小的路径扰动放大成巨大的相位。

除了频率，另一个关键变量是**基线长度**。天气屏在短基线两端看起来比较相似，在长基线两端则几乎完全不相关，因此相干度会随基线增长而下降。


In [ ]:
baseline_m = np.logspace(2, 5, 240)
sigma_path_m = 0.01e-3 * (baseline_m / 100.0) ** 0.83

coherence_curves = {}
for nu_ghz in [22.0, 100.0, 230.0]:
    phase_rms = phase_from_path_delay_rad(sigma_path_m, nu_ghz * 1e9)
    coherence_curves[nu_ghz] = np.exp(-0.5 * phase_rms**2)

fig, ax = plt.subplots(figsize=(9, 4.8))
for nu_ghz, coherence in coherence_curves.items():
    ax.semilogx(baseline_m / 1e3, coherence, lw=2.0, label=f"{nu_ghz:.0f} GHz")
ax.set_xlabel("baseline [km]")
ax.set_ylabel("coherence loss factor")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Longer baselines suffer more from tropospheric decorrelation")
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)

print("当 coherence 首次低于 0.5 时的基线尺度：")
for nu_ghz, coherence in coherence_curves.items():
    below = np.where(coherence < 0.5)[0]
    limit_km = baseline_m[below[0]] / 1e3 if len(below) else None
    if limit_km is None:
        print(f"  {nu_ghz:6.1f} GHz -> 在给定基线范围内仍高于 0.5")
    else:
        print(f"  {nu_ghz:6.1f} GHz -> ≈ {limit_km:.2f} km")


### 7.7.4 对观测与校准的启示

传播效应并不是一串分散的小知识点，而是三类真正会主导观测策略的限制因素：

- **低频**：电离层相位、法拉第旋转和星际/电离层散射最麻烦，方向依赖校准往往是必需的；
- **厘米波到毫米波过渡区**：主波束、极化和传播误差会同时进入高动态范围成像问题；
- **毫米波与亚毫米波**：对流层湿延迟成为主角，快切换、水汽辐射计和严格天气筛选直接决定数据质量。

更重要的是，这些传播效应都会回到可见度层面：

- 散射改变的是相干函数与表观源结构；
- 电离层改变的是色散相位、群延迟和偏振旋转；
- 对流层改变的是近似非色散的路径长度和高频相干度。

因此，从校准的角度看，它们都应被理解为 RIME 链中的传播 Jones 项，而不是“成像结束后再补一个经验修正”的附属后处理。
